# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset loaded from: {croissant_url}\n")

# Access metadata attributes
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets and their structure by `@id`. This helps us understand data organization and plan extraction.

For each record set, we display its `@id` and summarize available fields.

In [ ]:
# List all record sets in the dataset via their @id
record_sets = []
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
else:
    # Try to infer them from all registered record sets in `dataset`
    record_sets = dataset.record_sets()

print("Available Record Sets (@id):")
for recset in record_sets:
    if hasattr(recset, '@id'):
        recset_id = recset['@id'] if isinstance(recset, dict) else recset.@id
    else:
        recset_id = recset
    print(f"- {recset_id}")

# For each record set, print its fields
for recset in record_sets:
    recset_id = recset['@id'] if isinstance(recset, dict) else (recset.@id if hasattr(recset, '@id') else recset)
    try:
        fields = dataset.fields(record_set=recset_id)
        print(f"\nFields in Record Set {recset_id} (referenced by @id):")
        for fld in fields:
            fid = fld['@id'] if isinstance(fld, dict) else fld.@id
            print(f"    * {fid}")
    except Exception as e:
        print(f"Could not list fields for record set {recset_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Identify record set and field `@id`s from the overview above.

In [ ]:
# Get list of all record set @ids
record_set_ids = []
for recset in record_sets:
    recset_id = recset['@id'] if isinstance(recset, dict) else (recset.@id if hasattr(recset, '@id') else recset)
    record_set_ids.append(recset_id)

# Load all record sets into pandas DataFrames
dataframes = {}
for recset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            dataframes[recset_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[recset_id])} records from record set '{recset_id}'. Columns: {dataframes[recset_id].columns.tolist()}")
        else:
            print(f"No records loaded for record set '{recset_id}'.")
    except Exception as e:
        print(f"Could not load records from record set '{recset_id}': {e}")

# For demonstration, show columns and head of the first available record set
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head(10)
else:
    print("No dataframes loaded. Please check the schema and try again.")

## 4. Exploratory Data Analysis (EDA)
We'll apply initial data analysis steps, such as filtering, normalizing numeric fields, and exploring categorical distributions.

> **Note**: All fields are referenced by their Croissant `@id`. Please replace values in variable assignment if you wish to analyze a different numeric or grouping field.

In [ ]:
# Example: Let's choose the main record set for EDA
if dataframes:
    df = dataframes[main_rs_id]
    print(f"Performing EDA on record set '{main_rs_id}'. Available fields (@id): {df.columns.tolist()}")

    # Choose a numeric field by @id (e.g. age at diagnosis, if present)
    # Replace 'age_at_second_crc_dx' with the real field @id if different
    numeric_field = None
    candidate_fields = [col for col in df.columns if ("age" in col.lower() or "interval" in col.lower() or "size" in col.lower())]
    if candidate_fields:
        numeric_field = candidate_fields[0]
    else:
        numeric_field = df.columns[0]  # fallback to first column

    print(f"\nUsing numeric field: {numeric_field}\n")

    # Try to convert to numeric if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10  # Use mean as threshold if meaningful

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing top 5):")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std if std else filtered_df[numeric_field]
    print(f"\nNormalized {numeric_field}: (showing top 5)")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Choose a group field, e.g. anatomical location or msi_status by @id, if in columns
    group_candidates = [col for col in df.columns if any(k in col.lower() for k in ["anatomy", "site", "msi", "sex", "group"])]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by field: {group_field}")
    else:
        group_field = None

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['count', 'mean', 'std', 'min', 'max'])
        print(f"\nGrouped statistics by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found for grouped statistics.")
else:
    print("No data available for EDA.")

## 5. Visualization
We visualize the distribution of the chosen numeric field and its relationship to a grouping field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If there is a group field
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load, examine, and analyze the FAIR² colorectal cancer survivors dataset using robust schema-based referencing by `@id`. We demonstrated how to:

- Load Croissant metadata and discover record sets and fields by `@id`.
- Extract tabular data directly into pandas DataFrames.
- Apply basic data filtering, normalization, and aggregation operations by referencing record set and field `@id`.
- Visualize distributions and relationships present in the dataset.

This provides a starting point for further reproducible and FAIR-aligned biomedical data science. For advanced tasks such as hypothesis testing, predictive modeling, or publication, expand these workflows and consult the dataset documentation and schema for precise field usage.